# RLAIF



In [ ]:
!pip -q install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.5 MB/s eta 0:00:00


In [ ]:
import os
import math
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ["OPENAI_API_KEY"] is not None
print("API key loaded")

API key loaded


In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
prompt = "초등학생에게 블랙홀이 무엇인지 3~4문장으로 쉽게 설명해줘."

def generate_answer(temp):
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful teacher."},
            {"role": "user", "content": prompt}
        ],
        temperature=temp,
    )
    return completion.choices[0].message.content


# 서로 다른 스타일의 답을 만들기 위해 temperature만 바꿈
A = generate_answer(0.2)
B = generate_answer(1.2)

print("=== Candidate A ===")
print(A)

print("\n=== Candidate B ===")
print(B)


=== Candidate A ===
블랙홀은 우주에 있는 아주 강력한 빨아들이는 힘을 가진 곳이야. 이 힘은 너무 강해서 빛조차 빠져나올 수 없어. 블랙홀은 별이 죽을 때 생길 수 있는데, 그 별이 아주 작고 무겁게 압축되면서 만들어져. 그래서 블랙홀 근처에 가면 모든 것이 그 안으로 빨려 들어가게 돼.

=== Candidate B ===
블랙홀은 우주에 있는 매우 무거운 물질 덩어리로, 모든 것을 강하게 끌어당기는 힘이 있어. 빛조차도 빠져나올 수 없어서 우리가 볼 수 없지만, 주변의 별과 가스가 블랙홀에 빨려 들어가는 모습을 통해 그 존재를 알 수 있어. 아주 먼 우주 탐험처럼 신비롭고 흥미로운 곳이지!


In [ ]:
criteria = (
  "Evaluate: correctness, age-appropriateness, clarity, conciseness.\n"
  "If both are good, prefer the one that is more accurate and avoids advanced jargon.\n"
)


def judge_once(user_prompt: str, A: str, B: str, criteria: str):
    judge_system = (
        "You are a strict evaluator.\n"
        "Choose the better answer.\n"
        "Reply with ONLY one letter: A or B."
    )

    judge_input = f"""[User Prompt]
{user_prompt}

[Criteria]
{criteria}

[Answer A]
{A}

[Answer B]
{B}

Which answer is better?
"""

    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": judge_system},
            {"role": "user", "content": judge_input}
        ],
        logprobs=True, # 모델이 A 또는 B라는 토큰을 선택할 확률 (llm모델의 속마음)
        top_logprobs=5
    )

    choice = completion.choices[0].message.content.strip().upper()
    logprob_data = completion.choices[0].logprobs

    # 마지막 토큰에서 A/B 확률 찾기
    top = logprob_data.content[-1].top_logprobs

    prob_A = None
    prob_B = None

    for t in top:
        if t.token.strip() == "A":
            prob_A = math.exp(t.logprob)
        if t.token.strip() == "B":
            prob_B = math.exp(t.logprob)

    return {
        "choice": choice,
        "prob_A": prob_A,
        "prob_B": prob_B
    }


In [ ]:
import math

def _canon(tok: str) -> str:
    return tok.replace("\n", "").strip().upper()

def judge_once(user_prompt: str, A: str, B: str, criteria: str, debug_print: bool = False):
    judge_system = (
        "You are a strict evaluator.\n"
        "Choose the better answer.\n"
        "Reply with ONLY one letter: A or B."
    )

    judge_input = f"""[User Prompt]
{user_prompt}

[Criteria]
{criteria}

[Answer A]
{A}

[Answer B]
{B}

Which answer is better?
"""

    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": judge_system},
            {"role": "user", "content": judge_input},
        ],
        temperature=0,
        max_tokens=1,
        stop=["\n"],
        logprobs=True,
        top_logprobs=20,
    )

    lp0 = completion.choices[0].logprobs.content[0]

    raw_choice_token = lp0.token              # 원문 토큰
    choice = _canon(raw_choice_token)         # "A" or "B" 로 정규화
    chosen_prob = math.exp(lp0.logprob)

    # top20 후보를 전부 정규화해서 모아보기
    tops = []
    for t in (lp0.top_logprobs or []):
        tops.append({
            "raw": t.token,
            "canon": _canon(t.token),
            "p": math.exp(t.logprob),
        })

    # A/B 후보 찾기 (canon 기준)
    candA = next((x for x in tops if x["canon"] == "A"), None)
    candB = next((x for x in tops if x["canon"] == "B"), None)

    prob_A = None
    prob_B = None
    if candA is not None and candB is not None:
        s = candA["p"] + candB["p"]
        prob_A = candA["p"] / s
        prob_B = candB["p"] / s

    # 모순 체크: choice가 A인데 prob_A < prob_B면 이상징후로 표시
    contradiction = None
    if prob_A is not None and prob_B is not None and choice in ("A", "B"):
        if choice == "A" and prob_A < prob_B:
            contradiction = True
        elif choice == "B" and prob_B < prob_A:
            contradiction = True
        else:
            contradiction = False

    if debug_print:
        print("RAW choice token:", repr(raw_choice_token))
        print("CANON choice:", choice, "chosen_prob:", chosen_prob)
        print("Top candidates (canon==A/B):", candA, candB)
        print("Top 10 tokens by p:")
        for x in sorted(tops, key=lambda z: z["p"], reverse=True)[:10]:
            print(" ", repr(x["raw"]), "->", x["canon"], f"{x['p']:.6f}")
        print("A/B normalized:", prob_A, prob_B, "contradiction:", contradiction)

    return {
        "choice": choice,
        "chosen_prob": chosen_prob,
        "prob_A": prob_A,
        "prob_B": prob_B,
        "contradiction": contradiction,
        "raw_choice_token": raw_choice_token,
        "ab_found": (candA is not None and candB is not None),
    }


In [ ]:
def judge_debiased(user_prompt: str, A: str, B: str, criteria: str):
    w1 = judge_once(user_prompt, A, B, criteria)
    w2 = judge_once(user_prompt, B, A, criteria)  # swap 질문


    # PASS1 (정상 기준)
    c1 = w1["choice"]
    p1_A = w1["prob_A"]
    p1_B = w1["prob_B"]

    # PASS2 → 원래 기준으로 복원
    c2 = w2["choice"]
    if c2 == "A":
        c2 = "B"
    elif c2 == "B":
        c2 = "A"

    # 확률도 뒤집기
    p2_A = w2["prob_B"]
    p2_B = w2["prob_A"]

    # voting
    votes = {"A": 0, "B": 0}
    if c1 in votes: votes[c1] += 1
    if c2 in votes: votes[c2] += 1

    if votes["A"] > votes["B"]:
        final = "A"
    elif votes["B"] > votes["A"]:
        final = "B"
    else:
        final = "TIE"

    # soft preference
    A_soft = (p1_A + p2_A) / 2
    B_soft = (p1_B + p2_B) / 2

    total = A_soft + B_soft
    if total > 0:
        A_soft /= total
        B_soft /= total

    return {
        "pass1": w1,
        "pass2_raw": w2,
        "votes": votes,
        "final": final,
        "soft_preference": {
            "A": A_soft,
            "B": B_soft
        }
    }

res = judge_debiased(prompt, A, B, criteria)


In [ ]:
print(res)


{'pass1': {'choice': 'A', 'chosen_prob': 0.9706870301020154, 'prob_A': 0.9706877659097766, 'prob_B': 0.029312234090223377, 'contradiction': False, 'raw_choice_token': 'A', 'ab_found': True}, 'pass2_raw': {'choice': 'A', 'chosen_prob': 0.6791776844381462, 'prob_A': 0.6791787056691698, 'prob_B': 0.32082129433083023, 'contradiction': False, 'raw_choice_token': 'A', 'ab_found': True}, 'votes': {'A': 1, 'B': 1}, 'final': 'TIE', 'soft_preference': {'A': 0.6457545301203034, 'B': 0.35424546987969663}}


In [ ]:
print("\n=== Debiased Judge Result ===\n")

p1 = res["pass1"]
p2 = res["pass2_raw"]

print(f"PASS 1 → choice: {p1['choice']}")
print(f"  P(A): {p1['prob_A']:.4f}")
print(f"  P(B): {p1['prob_B']:.4f}")

print("\nPASS 2 (swap 질문)")
print(f"  P(A): {p2['prob_B']:.4f}")
print(f"  P(B): {p2['prob_A']:.4f}")

print("\nVotes:")
print(f"  A: {res['votes']['A']}")
print(f"  B: {res['votes']['B']}")

print(f"\n🏆 Final Winner: {res['final']}")

soft = res["soft_preference"]
print("\nSoft Preference (after debias):")
print(f"  A: {soft['A']*100:.1f}%")
print(f"  B: {soft['B']*100:.1f}%")



=== Debiased Judge Result ===

PASS 1 → choice: A
  P(A): 0.9707
  P(B): 0.0293

PASS 2 (swap 질문)
  P(A): 0.3208
  P(B): 0.6792

Votes:
  A: 1
  B: 1

🏆 Final Winner: TIE

Soft Preference (after debias):
  A: 64.6%
  B: 35.4%
